# DIM_TEMPO

Calendário diário de **2000-01-01 a 2030-12-31**, gerado no próprio banco (`CONNECT BY`).
A chave é inteligente, no formato `AAAAMMDD` (ex.: `20240310`). Nomes de mês e de dia
da semana em português.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_TEMPO d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_TEMPO = s.SK)
WHEN NOT MATCHED THEN INSERT
    (SK_TEMPO, DT_COMPLETA, NR_DIA, NR_MES, NOME_MES, ANO_MES, NR_TRIMESTRE, NR_SEMESTRE,
     NR_ANO, NOME_DIA_SEMANA, FLG_FIM_SEMANA)
VALUES (-1, NULL, NULL, NULL, 'NAO INFORMADO', 'N/I', NULL, NULL, NULL, 'NAO INFORMADO', 'N')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_TEMPO d
USING (
    SELECT TO_NUMBER(TO_CHAR(DT, 'YYYYMMDD'))                       AS SK_TEMPO,
           DT                                                        AS DT_COMPLETA,
           EXTRACT(DAY FROM DT)                                      AS NR_DIA,
           EXTRACT(MONTH FROM DT)                                    AS NR_MES,
           TO_CHAR(DT, 'fmMONTH', 'NLS_DATE_LANGUAGE=PORTUGUESE')    AS NOME_MES,
           TO_CHAR(DT, 'YYYY-MM')                                    AS ANO_MES,
           TO_NUMBER(TO_CHAR(DT, 'Q'))                               AS NR_TRIMESTRE,
           CASE WHEN EXTRACT(MONTH FROM DT) <= 6 THEN 1 ELSE 2 END   AS NR_SEMESTRE,
           EXTRACT(YEAR FROM DT)                                     AS NR_ANO,
           TO_CHAR(DT, 'fmDAY', 'NLS_DATE_LANGUAGE=PORTUGUESE')      AS NOME_DIA_SEMANA,
           CASE WHEN DT - TRUNC(DT, 'IW') >= 5 THEN 'S' ELSE 'N' END AS FLG_FIM_SEMANA
    FROM (SELECT DATE '2000-01-01' + LEVEL - 1 AS DT
          FROM DUAL
          CONNECT BY LEVEL <= DATE '2030-12-31' - DATE '2000-01-01' + 1)
) s
ON (d.SK_TEMPO = s.SK_TEMPO)
WHEN NOT MATCHED THEN INSERT
    (SK_TEMPO, DT_COMPLETA, NR_DIA, NR_MES, NOME_MES, ANO_MES, NR_TRIMESTRE, NR_SEMESTRE,
     NR_ANO, NOME_DIA_SEMANA, FLG_FIM_SEMANA)
VALUES
    (s.SK_TEMPO, s.DT_COMPLETA, s.NR_DIA, s.NR_MES, s.NOME_MES, s.ANO_MES, s.NR_TRIMESTRE,
     s.NR_SEMESTRE, s.NR_ANO, s.NOME_DIA_SEMANA, s.FLG_FIM_SEMANA)
"""

In [ ]:
mesclar("DIM_TEMPO", SQL, origem="(SELECT LEVEL FROM DUAL CONNECT BY LEVEL <= DATE '2030-12-31' - DATE '2000-01-01' + 1)")

In [ ]:
consultar("SELECT * FROM DIM_TEMPO ORDER BY 1 FETCH FIRST 10 ROWS ONLY")